# Apri MozartAI

Esegui la cella qui sotto e apri il link `gradio.live` che compare nell'output. Lascia la cella in esecuzione mentre usi l'app.

La modalità pubblica scarica soltanto il checkpoint condiviso dal progetto. Finché il progetto non ha configurato un ID pubblico, puoi avviare la demo dal tuo Drive selezionando **usa_drive_personale** se lì hai già `best.pt`.


In [ ]:
#@title Avvia MozartAI
usa_drive_personale = False #@param {type:"boolean"}

import importlib.util
import os
from pathlib import Path
import subprocess
import sys
import tempfile

# Il responsabile del progetto può inserire l'ID del proprio best.pt
# condiviso in sola lettura con "Chiunque abbia il link".
ID_CHECKPOINT_PUBBLICO = ""
MODELLO_DRIVE = Path("/content/drive/MyDrive/MozartAI/checkpoints_v2/best.pt")

if not usa_drive_personale and not ID_CHECKPOINT_PUBBLICO:
    raise RuntimeError(
        "La demo pubblica aspetta il checkpoint best.pt del progetto. "
        "Se hai già il file nel tuo Drive, seleziona usa_drive_personale e riesegui la cella."
    )

cartella = Path(tempfile.mkdtemp(prefix="mozartai-"))
repo = cartella / "MozartAI"

if usa_drive_personale:
    from google.colab import drive

    drive.mount("/content/drive")
    modello = MODELLO_DRIVE
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown

    modello = cartella / "best.pt"
    if not gdown.download(id=ID_CHECKPOINT_PUBBLICO, output=str(modello), quiet=False):
        raise RuntimeError("Download di best.pt fallito. Controlla la condivisione del checkpoint.")

if not modello.is_file() or modello.stat().st_size == 0:
    raise FileNotFoundError(f"Checkpoint best.pt mancante o vuoto: {modello}")

subprocess.run([
    "git", "clone", "--depth", "1", "--branch", "main",
    "https://github.com/PaoloniCoder/MozartAI.git", str(repo),
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "gradio==6.32.0", "pretty_midi", "pydantic>=2.12.5,<3", "starlette>=1.3.1,<2",
], check=True)

os.environ["MOZART_MODELLO"] = str(modello)
spec = importlib.util.spec_from_file_location("mozart_app", repo / "app.py")
mozart_app = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mozart_app)

print("Modello:", mozart_app.precarica())
mozart_app.avvia_app(share=True, debug=True)
